# EDA — Análisis Exploratorio de Datos

## BEKANTOR Demand Intelligence

El objetivo de esta etapa es explorar el comportamiento de las ventas,
detectar patrones, relaciones, valores atípicos y posibles problemas
que deberán tratarse posteriormente durante la limpieza profunda y
la ingeniería de variables.

## Librerías y configuración

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

### Conexión del notebook con el proyecto

In [ ]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT) 

In [ ]:
from src.data.load_data import load_raw_data
from src.data.clean_data import minimal_cleaning

datasets = load_raw_data()
datasets = minimal_cleaning(datasets)

sales = datasets["sales"]
stores = datasets["stores"]
transactions = datasets["transactions"]
holidays = datasets["holidays"]
oil = datasets["oil"]

In [ ]:
print(f"Filas oil después de completar calendario: {len(oil):,}")
print(f"Nulos restantes: {oil['dcoilwtico'].isna().sum()}")
print(f"Desde: {oil['date'].min().date()}")
print(f"Hasta: {oil['date'].max().date()}")

In [ ]:
from src.data.panel_diagnostics import (
    add_store_operation_flags,
    add_zero_taxonomy,
    find_missing_dates
)

## 6.0 Diagnóstico estructural del panel temporal

Antes de analizar la distribución de las ventas se verifica la estructura temporal del panel, las fechas de apertura de las tiendas y la naturaleza de los registros con ventas iguales a cero.

In [ ]:
sales, apertura = add_store_operation_flags(sales)

apertura.sort_values()

In [ ]:
dataset_start = sales["date"].min()

late_openings = apertura[
    apertura > pd.Timestamp("2013-01-02")
].sort_values()

late_openings

In [ ]:
pre_opening = (
    (sales["sales"] == 0) &
    (~sales["tienda_operativa"])
)

print(f"Ceros pre-apertura: {pre_opening.sum():,}")
print(
    f"Porcentaje del dataset: "
    f"{pre_opening.mean() * 100:.2f}%"
)

In [ ]:
sales = add_zero_taxonomy(sales)

In [ ]:
zero_summary = (
    sales["tipo_cero"]
    .value_counts()
    .rename_axis("tipo")
    .reset_index(name="registros")
)

zero_summary["porcentaje"] = (
    zero_summary["registros"]
    / len(sales)
    * 100
)

zero_summary

In [ ]:
missing_dates = find_missing_dates(sales)

print(f"Cantidad de fechas faltantes: {len(missing_dates)}")

for date in missing_dates:
    print(date.date())

In [ ]:
# Panel operativo: se conserva para diagnósticos estructurales
sales_valid = sales[
    sales["tienda_operativa"]
].copy()

# Panel modelable: base de todos los análisis de demanda
sales_modelable = sales[
    sales["tienda_operativa"] &
    sales["familia_surtida"]
].copy()

print(f"Panel original: {len(sales):,}")
print(f"Panel operativo: {len(sales_valid):,}")
print(f"Panel modelable: {len(sales_modelable):,}")


## 6.1 Distribución de las ventas

In [ ]:
sales_modelable.head()

In [ ]:
# Resumen sobre el panel realmente modelable
sales_modelable[["sales", "onpromotion"]].describe()

In [ ]:
total_modelable = len(sales_modelable)
zero_modelable = (sales_modelable["sales"] == 0).sum()
zero_pct_modelable = zero_modelable / total_modelable * 100

print(f"Registros modelables: {total_modelable:,}")
print(f"Tiendas: {sales_modelable['store_nbr'].nunique()}")
print(f"Familias: {sales_modelable['family'].nunique()}")
print(f"Ceros reales de demanda: {zero_modelable:,}")
print(f"Porcentaje de ceros reales: {zero_pct_modelable:.2f}%")
print(f"Ventas negativas: {(sales_modelable['sales'] < 0).sum():,}")

In [ ]:
zero_comparison = pd.DataFrame({
    "panel": [
        "Original",
        "Operativo",
        "Modelable"
    ],
    "registros": [
        len(sales),
        len(sales_valid),
        len(sales_modelable)
    ],
    "ceros": [
        (sales["sales"] == 0).sum(),
        (sales_valid["sales"] == 0).sum(),
        (sales_modelable["sales"] == 0).sum()
    ]
})

zero_comparison["porcentaje_ceros"] = (
    zero_comparison["ceros"]
    / zero_comparison["registros"]
    * 100
)

zero_comparison

In [ ]:
fig = px.bar(
    zero_comparison,
    x="panel",
    y="porcentaje_ceros",
    text="porcentaje_ceros",
    title="Impacto de los ceros estructurales sobre el panel"
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Panel",
    yaxis_title="% de registros con sales = 0"
)

fig.show()

In [ ]:
sales_modelable["sales"].quantile([
    0,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
    1
])

### Distribución de ventas (muestra)

In [ ]:
sales_sample = sales_modelable.sample(
    n=min(200_000, len(sales_modelable)),
    random_state=42
)

p99 = sales_modelable["sales"].quantile(0.99)

sales_visual = sales_sample[
    sales_sample["sales"] <= p99
]

In [ ]:
fig = px.histogram(
    sales_visual,
    x="sales",
    nbins=100,
    title="Distribución de ventas — panel modelable (hasta P99)"
)

fig.update_layout(
    xaxis_title="Ventas",
    yaxis_title="Cantidad de registros"
)

fig.show()

In [ ]:
zero_sales_by_family = (
    sales_modelable
    .assign(is_zero=sales_modelable["sales"] == 0)
    .groupby("family", as_index=False)
    .agg(
        registros=("sales", "size"),
        ventas_cero=("is_zero", "sum")
    )
)

zero_sales_by_family["pct_zero"] = (
    zero_sales_by_family["ventas_cero"]
    / zero_sales_by_family["registros"]
    * 100
)

zero_sales_by_family = (
    zero_sales_by_family
    .sort_values("pct_zero", ascending=False)
)

zero_sales_by_family.head(15)

In [ ]:
fig = px.bar(
    zero_sales_by_family.head(15),
    x="pct_zero",
    y="family",
    orientation="h",
    title="Familias con mayor proporción de ceros reales de demanda"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    xaxis_title="% de ceros reales",
    yaxis_title="Familia"
)

fig.show()

### Hallazgo — estructura de los ceros

El análisis estructural muestra que los registros con `sales = 0` no representan un único fenómeno.

En el panel original, el 31,30 % de las observaciones presentaban ventas iguales a cero. Sin embargo, se identificaron **222.057 registros pre-apertura**, equivalentes al **7,40 % del dataset**, y **81.624 registros correspondientes a familias que determinadas tiendas no comercializan**.

Luego de excluir estos ceros estructurales, el panel modelable queda compuesto por **2.697.207 observaciones**, de las cuales **635.449 presentan ausencia real de ventas**, equivalente al **23,56 % del panel modelable**.

La demanda cero tampoco se distribuye uniformemente entre categorías: familias como `BOOKS` y `BABY CARE` presentan más del **90 % de registros sin ventas**, evidenciando un comportamiento altamente intermitente.

Esta separación evita que las etapas posteriores interpreten como baja demanda situaciones en las que la tienda todavía no estaba operativa o la familia de productos no formaba parte de su surtido.

## 6.2 Evolución temporal: crecimiento vs. expansión comercial

La evolución agregada puede crecer tanto porque venden más las tiendas existentes como porque se incorporan nuevas tiendas. Para no confundir ambos efectos se separan tres componentes: ventas de tiendas comparables, cantidad de tiendas activas y ventas promedio por tienda activa.

In [ ]:
# Tiendas comparables: operativas prácticamente desde el inicio del período
comparable_stores = apertura[
    apertura <= pd.Timestamp("2013-01-02")
].index

modelable_monthly = sales_modelable.copy()
modelable_monthly["month"] = modelable_monthly["date"].dt.to_period("M").dt.to_timestamp()

# 1) Ventas mensuales de una base fija de tiendas comparables
comparable_monthly = (
    modelable_monthly[
        modelable_monthly["store_nbr"].isin(comparable_stores)
    ]
    .groupby("month", as_index=False)["sales"]
    .sum()
    .rename(columns={"sales": "ventas_tiendas_comparables"})
)

# 2) Cantidad de tiendas activas por mes
active_store_month = (
    sales_modelable
    .groupby([sales_modelable["date"].dt.to_period("M"), "store_nbr"])
    .size()
    .reset_index(name="registros")
    .rename(columns={"date": "month_period"})
)
active_store_month["month"] = active_store_month["month_period"].dt.to_timestamp()
active_store_count = (
    active_store_month
    .groupby("month", as_index=False)["store_nbr"]
    .nunique()
    .rename(columns={"store_nbr": "tiendas_activas"})
)

# 3) Venta diaria promedio por tienda activa dentro de cada mes
store_daily = (
    sales_modelable
    .groupby(["date", "store_nbr"], as_index=False)["sales"]
    .sum()
)
store_daily["month"] = store_daily["date"].dt.to_period("M").dt.to_timestamp()
avg_active_store = (
    store_daily
    .groupby("month", as_index=False)["sales"]
    .mean()
    .rename(columns={"sales": "venta_diaria_promedio_tienda_activa"})
)

growth_decomposition = (
    comparable_monthly
    .merge(active_store_count, on="month", how="inner")
    .merge(avg_active_store, on="month", how="inner")
)

# Agosto de 2017 es parcial: se excluye de la comparación mensual.
growth_decomposition = growth_decomposition[
    growth_decomposition["month"] < "2017-08-01"
].copy()

growth_decomposition.tail()

In [ ]:
# Índices base 100 para comparar magnitudes con escalas diferentes en una sola figura.
indexed_growth = growth_decomposition.copy()
metrics = [
    "ventas_tiendas_comparables",
    "tiendas_activas",
    "venta_diaria_promedio_tienda_activa",
]

for metric in metrics:
    indexed_growth[f"{metric}_indice"] = (
        indexed_growth[metric]
        / indexed_growth[metric].iloc[0]
        * 100
    )

plot_growth = indexed_growth.melt(
    id_vars="month",
    value_vars=[f"{m}_indice" for m in metrics],
    var_name="componente",
    value_name="indice_base_100",
)

labels = {
    "ventas_tiendas_comparables_indice": "Ventas tiendas comparables",
    "tiendas_activas_indice": "Cantidad de tiendas activas",
    "venta_diaria_promedio_tienda_activa_indice": "Venta diaria promedio por tienda activa",
}
plot_growth["componente"] = plot_growth["componente"].map(labels)

fig = px.line(
    plot_growth,
    x="month",
    y="indice_base_100",
    color="componente",
    title="Crecimiento real vs. expansión comercial — índice base 100",
)
fig.update_layout(
    xaxis_title="Mes",
    yaxis_title="Índice (inicio = 100)",
    legend_title="Componente",
)
fig.show()

### Hallazgo — crecimiento y expansión

La tendencia agregada no debe interpretarse directamente como crecimiento de la demanda. La descomposición permite observar por separado el desempeño de una base fija de tiendas, la expansión del número de establecimientos activos y la venta promedio por tienda activa. De esta manera, el crecimiento comercial deja de confundirse con la apertura progresiva de nuevas tiendas.

## 6.3 Ventas por familia de producto

In [ ]:
# Ventas totales por familia
sales_by_family = (
    sales_modelable
    .groupby("family", as_index=False)["sales"]
    .sum()
    .sort_values("sales", ascending=False)
)

sales_by_family.head(10)

In [ ]:
fig = px.bar(
    sales_by_family.head(15),
    x="sales",
    y="family",
    orientation="h",
    title="Top 15 familias por ventas totales"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

In [ ]:
zero_sales_by_family = (
    sales_modelable
    .assign(is_zero=sales_modelable["sales"] == 0)
    .groupby("family", as_index=False)
    .agg(
        registros=("sales", "size"),
        ventas_cero=("is_zero", "sum")
    )
)

zero_sales_by_family["pct_zero"] = (
    zero_sales_by_family["ventas_cero"]
    / zero_sales_by_family["registros"]
    * 100
)

zero_sales_by_family = zero_sales_by_family.sort_values(
    "pct_zero",
    ascending=False
)

zero_sales_by_family.head(10)

In [ ]:
fig = px.bar(
    zero_sales_by_family.head(15),
    x="pct_zero",
    y="family",
    orientation="h",
    title="Familias con mayor porcentaje de registros sin ventas"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

Conclusión por familia: La demanda se encuentra fuertemente concentrada en pocas familias de productos, especialmente GROCERY I, BEVERAGES y PRODUCE. Al mismo tiempo, existen familias con una elevada proporción de registros sin ventas, lo que evidencia patrones de demanda intermitente. Esto deberá considerarse posteriormente tanto en la ingeniería de variables como en la construcción del modelo predictivo.

## 6.4 Ventas por tienda — comparación por día operativo

Los totales históricos favorecen a las tiendas que estuvieron abiertas durante más tiempo. Para realizar una comparación más justa se utiliza la **venta diaria promedio por día operativo** de cada establecimiento sobre el panel modelable.

In [ ]:
store_daily_modelable = (
    sales_modelable
    .groupby(["store_nbr", "date"], as_index=False)["sales"]
    .sum()
)

store_performance = (
    store_daily_modelable
    .groupby("store_nbr", as_index=False)
    .agg(
        dias_operativos=("date", "nunique"),
        ventas_totales=("sales", "sum"),
        venta_diaria_promedio=("sales", "mean"),
        venta_diaria_mediana=("sales", "median"),
    )
    .merge(stores, on="store_nbr", how="left")
    .sort_values("venta_diaria_promedio", ascending=False)
)

store_performance.head(15)

In [ ]:
top_stores_avg = store_performance.head(15).copy()
top_stores_avg["store_nbr"] = top_stores_avg["store_nbr"].astype(str)

fig = px.bar(
    top_stores_avg.sort_values("venta_diaria_promedio"),
    x="venta_diaria_promedio",
    y="store_nbr",
    orientation="h",
    hover_data=["dias_operativos", "city", "type", "cluster"],
    title="Top 15 tiendas por venta diaria promedio — días operativos",
)
fig.update_layout(
    xaxis_title="Venta diaria promedio",
    yaxis_title="Tienda",
)
fig.show()

In [ ]:
store_type_profile = (
    store_performance
    .groupby("type", as_index=False)
    .agg(
        tiendas=("store_nbr", "nunique"),
        venta_diaria_promedio=("venta_diaria_promedio", "mean"),
        venta_diaria_mediana=("venta_diaria_promedio", "median"),
    )
    .sort_values("venta_diaria_promedio", ascending=False)
)

store_type_profile

### Hallazgo — comparación justa entre tiendas

La comparación se realiza sobre **ventas promedio por día operativo**, no sobre acumulados históricos. Así se reduce el sesgo que favorecía a las tiendas con mayor antigüedad y las diferencias observadas reflejan mejor el desempeño cotidiano de cada establecimiento.

## 6.5 Promociones y ventas — efecto controlado

El análisis inicial comparaba directamente todos los registros con y sin promoción. Sin embargo, `onpromotion` presenta una fuerte dependencia temporal y las promociones se concentran en determinadas familias de mayor volumen.

Para reducir este sesgo, se analiza primero la evolución temporal de las promociones y posteriormente se compara su efecto desde 2014 dentro de combinaciones tienda–familia.

In [ ]:
promo_year = (
    sales_modelable
    .assign(
        year=sales_modelable["date"].dt.year,
        has_promotion=sales_modelable["onpromotion"] > 0
    )
    .groupby("year", as_index=False)
    .agg(
        registros=("sales", "size"),
        registros_promo=("has_promotion", "sum"),
        ventas_promedio=("sales", "mean")
    )
)

promo_year["pct_promocion"] = (
    promo_year["registros_promo"]
    / promo_year["registros"]
    * 100
)

promo_year

In [ ]:
fig = px.bar(
    promo_year,
    x="year",
    y="pct_promocion",
    text="pct_promocion",
    title="Evolución de la presencia de promociones por año"
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Año",
    yaxis_title="% de registros con promoción"
)

fig.show()

In [ ]:
promo_data = sales_modelable[
    sales_modelable["date"] >= "2014-01-01"
].copy()

promo_data["has_promotion"] = (
    promo_data["onpromotion"] > 0
)

In [ ]:
no_promo = (
    promo_data[
        ~promo_data["has_promotion"]
    ]
    .groupby(
        ["store_nbr", "family"],
        as_index=False
    )
    .agg(
        n_sin_promo=("sales", "size"),
        ventas_sin_promo=("sales", "mean")
    )
)

In [ ]:
with_promo = (
    promo_data[
        promo_data["has_promotion"]
    ]
    .groupby(
        ["store_nbr", "family"],
        as_index=False
    )
    .agg(
        n_con_promo=("sales", "size"),
        ventas_con_promo=("sales", "mean")
    )
)

In [ ]:
promo_lift = no_promo.merge(
    with_promo,
    on=["store_nbr", "family"],
    how="inner"
)

In [ ]:
promo_lift = promo_lift[
    (promo_lift["n_sin_promo"] >= 20) &
    (promo_lift["n_con_promo"] >= 20) &
    (promo_lift["ventas_sin_promo"] > 0)
].copy()

In [ ]:
promo_lift["lift_ratio"] = (
    promo_lift["ventas_con_promo"]
    / promo_lift["ventas_sin_promo"]
)

In [ ]:
family_promo_lift = (
    promo_lift
    .groupby("family", as_index=False)
    .agg(
        tiendas_comparables=("store_nbr", "nunique"),
        lift_mediano=("lift_ratio", "median")
    )
    .sort_values(
        "lift_mediano",
        ascending=False
    )
)

family_promo_lift

In [ ]:
familias_principales = [
    "PRODUCE",
    "BEVERAGES",
    "DAIRY",
    "GROCERY I",
    "MEATS"
]

family_promo_lift[
    family_promo_lift["family"].isin(
        familias_principales
    )
].sort_values(
    "lift_mediano",
    ascending=False
)

In [ ]:
promo_plot = (
    family_promo_lift[
        family_promo_lift["family"].isin(
            familias_principales
        )
    ]
    .sort_values(
        "lift_mediano",
        ascending=True
    )
)

fig = px.bar(
    promo_plot,
    x="lift_mediano",
    y="family",
    orientation="h",
    text="lift_mediano",
    title="Lift promocional dentro de tienda–familia desde 2014"
)

fig.update_traces(
    texttemplate="%{text:.2f}×",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Ratio ventas con promoción / sin promoción",
    yaxis_title="Familia"
)

fig.show()

In [ ]:
from src.features.statistical_analysis import bootstrap_ci

promo_ci_rows = []
for family, group in promo_lift.groupby("family"):
    estimate, ci_low, ci_high, n_series = bootstrap_ci(
        group["lift_ratio"],
        statistic=np.median,
        n_boot=2000,
        confidence=0.95,
        random_state=42,
    )
    promo_ci_rows.append({
        "family": family,
        "series_comparables": n_series,
        "lift_mediano": estimate,
        "ci_95_low": ci_low,
        "ci_95_high": ci_high,
        "observaciones_con_promo": int(group["n_con_promo"].sum()),
        "observaciones_sin_promo": int(group["n_sin_promo"].sum()),
    })

promo_uncertainty = (
    pd.DataFrame(promo_ci_rows)
    .sort_values("lift_mediano", ascending=False)
)

promo_uncertainty.head(15)

### Hallazgo — efecto promocional controlado

La comparación global entre registros con y sin promoción estaba afectada por la evolución temporal de `onpromotion`. El análisis se realiza dentro de cada combinación tienda–familia desde 2014 y se interpreta como **asociación**, no como causalidad.

Además del lift mediano se informa el número de series y observaciones comparables y un **intervalo bootstrap del 95%**. Esto aporta contexto a lifts extremos: un valor elevado deja de evaluarse de forma aislada y se acompaña por su incertidumbre estadística.

Entre las familias principales continúan destacándose aproximadamente **PRODUCE 1,85×**, **BEVERAGES 1,54×** y **GROCERY I 1,24×**, ahora con su correspondiente intervalo de confianza.

## 6.6 Estacionalidad de la demanda

La demanda se analiza como una serie temporal para identificar patrones recurrentes asociados al calendario.

Se estudian tres componentes principales: día de la semana, día del mes y la interacción entre día de semana y mes del año.

Los días 1 de enero se excluyen de este análisis estacional debido a que representan días de cierre o actividad excepcionalmente baja y no un patrón normal de demanda.

In [ ]:
daily_modelable = (
    sales_modelable
    .groupby("date", as_index=False)["sales"]
    .sum()
)

daily_modelable["day_of_week"] = daily_modelable["date"].dt.dayofweek
daily_modelable["day_name"] = daily_modelable["date"].dt.day_name()
daily_modelable["day_of_month"] = daily_modelable["date"].dt.day
daily_modelable["month"] = daily_modelable["date"].dt.month
daily_modelable["year"] = daily_modelable["date"].dt.year

# Excluir 1 de enero: cierre / funcionamiento excepcional
seasonal_daily = daily_modelable[
    ~(
        (daily_modelable["date"].dt.month == 1) &
        (daily_modelable["date"].dt.day == 1)
    )
].copy()

In [ ]:
overall_daily_mean = seasonal_daily["sales"].mean()

weekday_index = (
    seasonal_daily
    .groupby(
        ["day_of_week", "day_name"],
        as_index=False
    )
    .agg(
        ventas_promedio=("sales", "mean")
    )
)

weekday_index["indice"] = (
    weekday_index["ventas_promedio"]
    / overall_daily_mean
)

weekday_index = weekday_index.sort_values("day_of_week")

weekday_index

In [ ]:
fig = px.bar(
    weekday_index,
    x="day_name",
    y="indice",
    text="indice",
    title="Índice de demanda por día de la semana"
)

fig.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside"
)

fig.add_hline(
    y=1,
    line_dash="dash"
)

fig.update_layout(
    xaxis_title="Día de la semana",
    yaxis_title="Índice vs. día promedio"
)

fig.show()

In [ ]:
weekday_baseline = (
    seasonal_daily
    .groupby("day_of_week")["sales"]
    .mean()
)

seasonal_daily["weekday_baseline"] = (
    seasonal_daily["day_of_week"]
    .map(weekday_baseline)
)

In [ ]:
seasonal_daily["sales_weekday_adjusted"] = (
    seasonal_daily["sales"]
    / seasonal_daily["weekday_baseline"]
)

In [ ]:
day_month_index = (
    seasonal_daily
    .groupby("day_of_month", as_index=False)
    .agg(
        indice=("sales_weekday_adjusted", "mean")
    )
)

day_month_index

In [ ]:
fig = px.line(
    day_month_index,
    x="day_of_month",
    y="indice",
    markers=True,
    title="Índice de demanda por día del mes — ajustado por día de semana"
)

fig.add_hline(
    y=1,
    line_dash="dash"
)

# Días relevantes
for day in [1, 16, 31]:
    fig.add_vline(
        x=day,
        line_dash="dot"
    )

fig.update_layout(
    xaxis_title="Día del mes",
    yaxis_title="Índice de demanda ajustado"
)

fig.show()

In [ ]:
seasonal_daily["year_month"] = (
    seasonal_daily["date"]
    .dt.to_period("M")
)

seasonal_daily["monthly_mean"] = (
    seasonal_daily
    .groupby("year_month")["sales"]
    .transform("mean")
)

seasonal_daily["monthly_index"] = (
    seasonal_daily["sales"]
    / seasonal_daily["monthly_mean"]
)

In [ ]:
heatmap_data = (
    seasonal_daily
    .groupby(
        ["month", "day_of_week"],
        as_index=False
    )
    .agg(
        indice=("monthly_index", "mean")
    )
)

In [ ]:
heatmap_pivot = heatmap_data.pivot(
    index="day_of_week",
    columns="month",
    values="indice"
)

day_labels = [
    "Lunes",
    "Martes",
    "Miércoles",
    "Jueves",
    "Viernes",
    "Sábado",
    "Domingo"
]

month_labels = [
    "Ene", "Feb", "Mar", "Abr",
    "May", "Jun", "Jul", "Ago",
    "Sep", "Oct", "Nov", "Dic"
]

heatmap_pivot.index = day_labels
heatmap_pivot.columns = month_labels

In [ ]:
fig = px.imshow(
    heatmap_pivot,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    title="Estacionalidad: día de semana × mes"
)

fig.update_layout(
    xaxis_title="Mes",
    yaxis_title="Día de la semana"
)

fig.show()

### Hallazgo — estacionalidad de la demanda

El análisis temporal evidencia una marcada estacionalidad asociada al calendario.

A nivel semanal, el **domingo presenta un índice de demanda de 1,30** y el **sábado de 1,21**, mientras que el **jueves desciende hasta 0,79**. Esto representa una diferencia relativa cercana al **63 % entre el día de mayor y menor demanda**, confirmando que el día de la semana constituye una señal predictiva relevante.

Luego de ajustar el efecto del día de semana, también aparece un patrón claro dentro del mes. El **día 1 registra un índice de 1,22**, seguido por valores elevados en los días 2 y 3. Posteriormente la demanda disminuye progresivamente, presenta un pequeño repunte alrededor del **día 16** y alcanza sus niveles más bajos entre los **días 25 y 28**, con índices cercanos a 0,91–0,93. Hacia el cierre del mes vuelve a observarse una recuperación.

Este comportamiento es compatible con un efecto de calendario asociado a comienzos de mes, quincena y períodos previos al cobro.

Finalmente, el heatmap día de semana × mes muestra que el patrón semanal es estable durante gran parte del año: los fines de semana, especialmente los domingos, presentan los mayores niveles relativos de demanda, mientras que los jueves muestran sistemáticamente los niveles más bajos.

Estos resultados justifican incorporar posteriormente variables de calendario como día de semana, día del mes, fin de semana y posición dentro del ciclo mensual.

## 6.7 Transacciones y ventas

Se analiza la relación entre el volumen diario de transacciones y las ventas agregadas por tienda.

Dado que ambas variables presentan tendencia y estacionalidad, la relación no se evalúa únicamente en niveles. También se estudian primeras diferencias y rezagos temporales para distinguir una asociación contemporánea de una posible relación predictiva.

In [ ]:
# Ventas diarias por tienda
sales_store_daily = (
    sales_modelable
    .groupby(["date", "store_nbr"], as_index=False)["sales"]
    .sum()
)

sales_transactions = (
    sales_store_daily
    .merge(
        transactions,
        on=["date", "store_nbr"],
        how="inner"
    )
    .sort_values(["store_nbr", "date"])
)

In [ ]:
corr_tx_levels = (
    sales_transactions["sales"]
    .corr(sales_transactions["transactions"])
)

print(
    f"Correlación en niveles: "
    f"{corr_tx_levels:.3f}"
)

In [ ]:
sales_transactions["gap_days"] = (
    sales_transactions
    .groupby("store_nbr")["date"]
    .diff()
    .dt.days
)

sales_transactions["diff_sales"] = (
    sales_transactions
    .groupby("store_nbr")["sales"]
    .diff()
)

sales_transactions["diff_transactions"] = (
    sales_transactions
    .groupby("store_nbr")["transactions"]
    .diff()
)

tx_diff = sales_transactions[
    sales_transactions["gap_days"] == 1
].dropna(
    subset=["diff_sales", "diff_transactions"]
).copy()

In [ ]:
corr_tx_diff = (
    tx_diff["diff_sales"]
    .corr(tx_diff["diff_transactions"])
)

print(
    f"Correlación primeras diferencias: "
    f"{corr_tx_diff:.3f}"
)

In [ ]:
tx_results = [
    {
        "comparacion": "Niveles",
        "correlacion": corr_tx_levels
    },
    {
        "comparacion": "Δ mismo día",
        "correlacion": corr_tx_diff
    }
]

for lag in [1, 7]:

    lagged = tx_diff[
        ["date", "store_nbr", "diff_transactions"]
    ].copy()

    lagged["date"] = (
        lagged["date"]
        + pd.Timedelta(days=lag)
    )

    lagged = lagged.rename(
        columns={
            "diff_transactions":
            f"diff_transactions_lag{lag}"
        }
    )

    comparison = tx_diff.merge(
        lagged,
        on=["date", "store_nbr"],
        how="inner"
    )

    corr = comparison["diff_sales"].corr(
        comparison[f"diff_transactions_lag{lag}"]
    )

    tx_results.append({
        "comparacion": f"Δ transacciones t-{lag}",
        "correlacion": corr
    })

tx_corr_summary = pd.DataFrame(tx_results)

tx_corr_summary

In [ ]:
fig = px.bar(
    tx_corr_summary,
    x="comparacion",
    y="correlacion",
    text="correlacion",
    title="Transacciones y ventas — niveles, diferencias y rezagos"
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Comparación",
    yaxis_title="Correlación"
)

fig.show()

In [ ]:
sales_transactions["transaction_group"] = pd.qcut(
    sales_transactions["transactions"],
    q=10,
    duplicates="drop"
)

transaction_summary = (
    sales_transactions
    .groupby("transaction_group", observed=True)
    .agg(
        avg_transactions=("transactions", "mean"),
        avg_sales=("sales", "mean")
    )
    .reset_index()
)

transaction_summary

In [ ]:
fig = px.line(
    transaction_summary,
    x="avg_transactions",
    y="avg_sales",
    markers=True,
    title="Ventas promedio según nivel de transacciones"
)

fig.show()

### Hallazgo — transacciones y ventas

En niveles, las ventas diarias por tienda presentan una correlación elevada con el número de transacciones (**r = 0,837**). Para verificar que esta asociación no estuviera explicada únicamente por tendencias compartidas, se repitió el análisis utilizando primeras diferencias.

La correlación entre los cambios diarios de ventas y transacciones se mantiene elevada (**r = 0,783**), indicando que ambas variables presentan una relación contemporánea importante incluso después de remover parte de su tendencia.

Al analizar rezagos, el cambio de transacciones del día anterior presenta una correlación prácticamente nula con el cambio de ventas actual (**r = 0,033**), mientras que el rezago de siete días conserva una asociación considerable (**r = 0,605**). Este resultado es consistente con la marcada estacionalidad semanal observada previamente.

Por lo tanto, las transacciones contienen información relevante sobre la actividad comercial, aunque las transacciones del mismo día no pueden utilizarse directamente para predecir ventas de ese mismo día si todavía no son conocidas al momento del pronóstico. En etapas posteriores deberán considerarse únicamente valores históricos o variables derivadas mediante rezagos.

## 6.8 Petróleo y ventas

El precio del petróleo se incorpora como variable externa de contexto macroeconómico.

Para evitar conclusiones espurias derivadas de tendencias compartidas, se compara su relación con las ventas tanto en niveles como en primeras diferencias y rezagos temporales.

Además, el precio se trata respetando el orden temporal de la información, utilizando únicamente valores conocidos hasta cada fecha.

In [ ]:
sales_daily_corr = (
    sales_modelable
    .groupby("date", as_index=False)["sales"]
    .sum()
)

sales_oil_corr = (
    sales_daily_corr
    .merge(
        oil,
        on="date",
        how="left"
    )
    .sort_values("date")
)

In [ ]:
corr_oil_levels = (
    sales_oil_corr["sales"]
    .corr(sales_oil_corr["dcoilwtico"])
)

print(
    f"Correlación en niveles: "
    f"{corr_oil_levels:.3f}"
)

In [ ]:
sales_oil_corr["gap_days"] = (
    sales_oil_corr["date"]
    .diff()
    .dt.days
)

sales_oil_corr["diff_sales"] = (
    sales_oil_corr["sales"].diff()
)

sales_oil_corr["diff_oil"] = (
    sales_oil_corr["dcoilwtico"].diff()
)

oil_diff = sales_oil_corr[
    sales_oil_corr["gap_days"] == 1
].dropna(
    subset=["diff_sales", "diff_oil"]
).copy()

In [ ]:
corr_oil_diff = (
    oil_diff["diff_sales"]
    .corr(oil_diff["diff_oil"])
)

print(
    f"Correlación primeras diferencias: "
    f"{corr_oil_diff:.3f}"
)

In [ ]:
oil_results = [
    {
        "comparacion": "Niveles",
        "correlacion": corr_oil_levels
    },
    {
        "comparacion": "Δ mismo día",
        "correlacion": corr_oil_diff
    }
]

for lag in [1, 7]:

    lagged = oil_diff[
        ["date", "diff_oil"]
    ].copy()

    lagged["date"] = (
        lagged["date"]
        + pd.Timedelta(days=lag)
    )

    lagged = lagged.rename(
        columns={
            "diff_oil":
            f"diff_oil_lag{lag}"
        }
    )

    comparison = oil_diff.merge(
        lagged,
        on="date",
        how="inner"
    )

    corr = comparison["diff_sales"].corr(
        comparison[f"diff_oil_lag{lag}"]
    )

    oil_results.append({
        "comparacion": f"Δ petróleo t-{lag}",
        "correlacion": corr
    })

oil_corr_summary = pd.DataFrame(oil_results)

oil_corr_summary

In [ ]:
fig = px.bar(
    oil_corr_summary,
    x="comparacion",
    y="correlacion",
    text="correlacion",
    title="Petróleo y ventas — niveles, diferencias y rezagos"
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Comparación",
    yaxis_title="Correlación"
)

fig.show()

In [ ]:
oil_time = sales_oil_corr.copy()

oil_time["sales_norm"] = (
    oil_time["sales"] / oil_time["sales"].max()
)

oil_time["oil_norm"] = (
    oil_time["dcoilwtico"] / oil_time["dcoilwtico"].max()
)

fig = px.line(
    oil_time,
    x="date",
    y=["sales_norm", "oil_norm"],
    title="Evolución relativa de ventas y precio del petróleo"
)

fig.show()

### Hallazgo — petróleo y ventas

Se analizó la relación entre el precio diario del petróleo y las ventas agregadas por fecha.

En niveles, ambas series presentan una correlación negativa moderada (**r = -0.627**). Sin embargo, este resultado puede estar influido por tendencias de largo plazo presentes en ambas variables y no necesariamente por una relación operativa directa.

Para verificarlo, se repitió el análisis utilizando primeras diferencias diarias. En este caso, la correlación entre cambios diarios de ventas y cambios diarios del precio del petróleo resulta prácticamente nula (**r = 0.025**).

También se evaluaron rezagos del petróleo. Tanto el rezago de un día (**r = 0.014**) como el de siete días (**r = 0.021**) muestran asociaciones despreciables con las variaciones de ventas.

En conjunto, estos resultados indican que el precio del petróleo no presenta una relación útil de corto plazo con la demanda observada en esta etapa del análisis. Por lo tanto, su aporte predictivo potencial sería limitado, al menos bajo estas transformaciones simples.

### Tratamiento temporal del precio del petróleo

El precio se reindexa al calendario diario y se completa únicamente mediante **forward fill (`ffill`)**, utilizando el último valor conocido. Se conserva **un único valor nulo el 2013-01-01**, porque no existe una cotización previa disponible. No se aplica `bfill` ni interpolación desde el futuro, ya que eso introduciría **data leakage**.

## 6.9 Outliers y anomalías temporales

La detección de anomalías se realiza sobre la serie temporal diaria.

El método IQR global utilizado inicialmente no resulta adecuado porque las ventas presentan tendencia y estacionalidad. Por este motivo se utiliza una referencia móvil basada en mediana y MAD, que compara cada día con su entorno temporal reciente.

In [ ]:
daily_anomaly = (
    sales_modelable
    .groupby("date", as_index=False)["sales"]
    .sum()
    .set_index("date")
    .reindex(
        pd.date_range(
            sales_modelable["date"].min(),
            sales_modelable["date"].max(),
            freq="D"
        )
    )
    .rename_axis("date")
)

daily_anomaly.head()

In [ ]:
rolling = daily_anomaly["sales"].rolling(
    window=28,
    center=True,
    min_periods=14
)

rolling_median = rolling.median()

In [ ]:
# 1. MAD corregido
rolling_mad = rolling.apply(
    lambda x: np.nanmedian(
        np.abs(x - np.nanmedian(x))
    ),
    raw=True
)

# 2. Recalcular robust_z
daily_anomaly["rolling_median"] = rolling_median

daily_anomaly["robust_z"] = (
    daily_anomaly["sales"]
    - daily_anomaly["rolling_median"]
) / (
    1.4826 * rolling_mad
)

# 3. Volver a detectar anomalías
anomalies = daily_anomaly[
    daily_anomaly["robust_z"].abs() > 4
].copy()

print(f"Anomalías detectadas: {len(anomalies)}")

anomalies[
    ["sales", "rolling_median", "robust_z"]
].sort_values("robust_z")

In [ ]:
jan1_check = daily_anomaly[
    (daily_anomaly.index.month == 1) &
    (daily_anomaly.index.day == 1)
][
    ["sales", "rolling_median", "robust_z"]
]

jan1_check

In [ ]:
fig = px.line(
    daily_anomaly.reset_index(),
    x="date",
    y="sales",
    title="Ventas diarias y anomalías temporales"
)

fig.add_scatter(
    x=anomalies.index,
    y=anomalies["sales"],
    mode="markers",
    name="Anomalías"
)

fig.show()

### Terremoto de Manabí — abril de 2016

El dataset de eventos registra el terremoto de Manabí en abril de 2016.

Se analiza este período como un posible quiebre estructural de corto plazo, comparando las ventas durante el evento y los días posteriores contra el promedio de las cuatro semanas anteriores.

In [ ]:
earthquake_events = holidays[
    holidays["description"]
    .str.contains(
        "Terremoto",
        case=False,
        na=False
    )
][
    ["date", "type", "description"]
]

earthquake_events

In [ ]:
earthquake_start = pd.Timestamp("2016-04-16")

baseline_start = earthquake_start - pd.Timedelta(days=28)
baseline_end = earthquake_start - pd.Timedelta(days=1)

baseline_sales = daily_anomaly.loc[
    baseline_start:baseline_end,
    "sales"
].mean()

print(
    f"Promedio diario 4 semanas previas: "
    f"{baseline_sales:,.0f}"
)

In [ ]:
earthquake_period = (
    daily_anomaly
    .loc["2016-04-16":"2016-04-30", ["sales"]]
    .copy()
)

earthquake_period["variacion_vs_previo"] = (
    earthquake_period["sales"]
    / baseline_sales
    - 1
) * 100

earthquake_period

In [ ]:
earthquake_period.loc[
    "2016-04-16":"2016-04-18"
]

In [ ]:
earthquake_plot = (
    daily_anomaly
    .loc["2016-03-19":"2016-05-01"]
    .reset_index()
)

fig = px.line(
    earthquake_plot,
    x="date",
    y="sales",
    title="Impacto del terremoto de Manabí sobre las ventas"
)

fig.add_hline(
    y=baseline_sales,
    line_dash="dash",
    annotation_text="Promedio 4 semanas previas"
)

fig.add_vline(
    x=pd.Timestamp("2016-04-16").timestamp() * 1000,
    line_dash="dot",
    annotation_text="Terremoto"
)

fig.show()

In [ ]:
# Incertidumbre del impacto inmediato del terremoto por serie tienda-familia.
quake_pre = sales_modelable[
    sales_modelable["date"].between("2016-03-19", "2016-04-15")
].groupby(["store_nbr", "family"], as_index=False).agg(
    pre_mean=("sales", "mean"),
    n_pre=("sales", "size"),
)

quake_post = sales_modelable[
    sales_modelable["date"].between("2016-04-17", "2016-04-18")
].groupby(["store_nbr", "family"], as_index=False).agg(
    post_mean=("sales", "mean"),
    n_post=("sales", "size"),
)

quake_series = quake_pre.merge(
    quake_post, on=["store_nbr", "family"], how="inner"
)
quake_series = quake_series[quake_series["pre_mean"] > 0].copy()
quake_series["lift"] = quake_series["post_mean"] / quake_series["pre_mean"]

quake_est, quake_low, quake_high, quake_n = bootstrap_ci(
    quake_series["lift"],
    statistic=np.median,
    n_boot=2000,
    confidence=0.95,
    random_state=42,
)

pd.DataFrame([{
    "series_comparables": quake_n,
    "lift_mediano_17_18_abril": quake_est,
    "ci_95_low": quake_low,
    "ci_95_high": quake_high,
}])

### Hallazgo — anomalías temporales y terremoto de Manabí

El método global basado en IQR fue reemplazado por una detección robusta mediante **mediana móvil de 28 días y MAD**, permitiendo comparar cada observación con su contexto temporal reciente.

Con un umbral de `|robust_z| > 4` se identificaron **21 días anómalos**, frente a los 5 detectados originalmente mediante IQR. Esto confirma que un umbral global resulta inadecuado para una serie con tendencia y estacionalidad.

Cuatro de los cinco días 1 de enero superan además el umbral estadístico de anomalía. El **1 de enero de 2015** no supera dicho umbral, pero igualmente se interpreta como un día de cierre o funcionamiento excepcional a partir del conocimiento del calendario.

También se identificó un quiebre relevante asociado al **terremoto de Manabí del 16 de abril de 2016**. Frente a un promedio de **761.229 ventas diarias durante las cuatro semanas anteriores**, las ventas aumentaron aproximadamente **13,3 % el 16 de abril, 67,1 % el 17 y 76,8 % el 18**, manteniéndose elevadas durante varios días posteriores.

Este evento constituye un ejemplo de shock externo capaz de modificar temporalmente el comportamiento habitual de la demanda y justifica incorporar eventos extraordinarios dentro del calendario analítico.

Como complemento, el efecto inmediato se resume por tienda–familia con un **lift mediano e intervalo bootstrap del 95%**, evitando interpretar los porcentajes agregados como una estimación exacta y sin incertidumbre.

## 6.10 Calendario completo de feriados y eventos

Los feriados y eventos no afectan necesariamente a todas las tiendas por igual.

Se construye un calendario con una fila por fecha y tienda, aplicando los eventos nacionales a todos los establecimientos, los regionales según el estado y los locales según la ciudad.

También se conservan los distintos tipos de evento y las fechas donde coinciden múltiples acontecimientos.

In [ ]:
from src.features.build_calendar import build_calendar

In [ ]:
calendar = build_calendar(
    sales,
    stores,
    holidays
)

calendar.head()

In [ ]:
print(f"Filas calendario: {len(calendar):,}")
print(f"Fechas: {calendar['date'].nunique():,}")
print(f"Tiendas: {calendar['store_nbr'].nunique()}")

In [ ]:
calendar[
    [
        "is_holiday",
        "is_event",
        "is_transfer",
        "is_additional",
        "is_bridge",
        "is_work_day"
    ]
].sum()

In [ ]:
multiple_events = calendar[
    calendar["event_count"] > 1
]

print(
    "Combinaciones fecha-tienda con múltiples eventos:",
    len(multiple_events)
)

multiple_events[
    [
        "date",
        "store_nbr",
        "city",
        "state",
        "event_count",
        "event_descriptions"
    ]
].head(20)

In [ ]:
calendar[
    calendar["date"].isin(missing_dates)
][
    [
        "date",
        "store_nbr",
        "city",
        "is_holiday",
        "is_event",
        "is_work_day"
    ]
].head(20)

In [ ]:
print(
    "Duplicados fecha-tienda:",
    calendar.duplicated(["date", "store_nbr"]).sum()
)

print(
    "Registros de cierre por Año Nuevo:",
    calendar["is_new_year_closure"].sum()
)

In [ ]:
# Efecto descriptivo de feriados sobre demanda modelable.
calendar_flags = calendar[[
    "date", "store_nbr", "is_holiday", "is_event",
    "is_new_year_closure"
]].copy()

holiday_sales = sales_modelable.merge(
    calendar_flags,
    on=["date", "store_nbr"],
    how="left",
)

# Año Nuevo se analiza como cierre estructural, no como un feriado comercial ordinario.
holiday_sales = holiday_sales[holiday_sales["is_new_year_closure"].eq(0)].copy()

holiday_series_rows = []
for (store_nbr, family), group in holiday_sales.groupby(["store_nbr", "family"]):
    h = group.loc[group["is_holiday"].eq(1), "sales"]
    n = group.loc[group["is_holiday"].eq(0), "sales"]
    if len(h) >= 5 and len(n) >= 20 and n.mean() > 0:
        holiday_series_rows.append({
            "store_nbr": store_nbr,
            "family": family,
            "n_holiday": len(h),
            "n_regular": len(n),
            "holiday_lift": h.mean() / n.mean(),
        })

holiday_series_lift = pd.DataFrame(holiday_series_rows)

holiday_est, holiday_low, holiday_high, holiday_n = bootstrap_ci(
    holiday_series_lift["holiday_lift"],
    statistic=np.median,
    n_boot=2000,
    confidence=0.95,
    random_state=42,
)

pd.DataFrame([{
    "series_comparables": holiday_n,
    "lift_mediano_feriado": holiday_est,
    "ci_95_low": holiday_low,
    "ci_95_high": holiday_high,
}])

### Hallazgo — calendario comercial por tienda

El calendario original de feriados no puede tratarse únicamente como una variable binaria por fecha, ya que existen eventos de diferente alcance territorial y múltiples acontecimientos pueden coincidir en un mismo día.

Se construyó un calendario completo de **91.152 combinaciones fecha–tienda**, correspondientes a **1.688 días y 54 establecimientos**, aplicando los eventos nacionales a todas las tiendas, los regionales según el estado y los locales según la ciudad.

Además, se conservaron por separado los tipos `Holiday`, `Event`, `Transfer`, `Additional`, `Bridge` y `Work Day`. Se identificaron **239 combinaciones fecha–tienda afectadas por más de un evento**, evitando la pérdida de información que producía el uso previo de `drop_duplicates("date")`.

El calendario también permite representar las cuatro fechas ausentes del panel de ventas —los 25 de diciembre de 2013 a 2016— y distinguir explícitamente los cierres asociados al 1 de enero.

Esta estructura deja preparado un insumo temporal a nivel tienda que podrá integrarse posteriormente al modelo sin asumir que todos los eventos afectan de igual forma a todos los establecimientos.

El efecto de feriados se estima únicamente sobre el **panel modelable**, excluyendo el cierre estructural de Año Nuevo, y se acompaña por un **intervalo bootstrap del 95%** sobre los lifts tienda–familia comparables.

## 6.11 Estructura de las series tienda–familia: ADI + CV²

La intermitencia se analiza exclusivamente sobre `sales_modelable`, de modo que los ceros por preapertura y familias no comercializadas permanecen fuera de esta clasificación.

En lugar de utilizar un umbral arbitrario de porcentaje de ceros, se aplican **ADI (Average Demand Interval)** y **CV² (coeficiente de variación cuadrado)** para distinguir demanda **regular, errática, intermitente y lumpy**.

In [ ]:
from src.features.demand_classification import (
    ADI_THRESHOLD,
    CV2_THRESHOLD,
    build_demand_profile,
)

sales_2017 = sales_modelable[
    sales_modelable["date"].dt.year == 2017
].copy()

series_2017 = build_demand_profile(sales_2017)

print(f"Umbral ADI: {ADI_THRESHOLD}")
print(f"Umbral CV²: {CV2_THRESHOLD}")
print(f"Series analizadas: {len(series_2017):,}")

series_2017.head()

In [ ]:
series_type_summary = (
    series_2017["tipo_demanda"]
    .value_counts()
    .rename_axis("tipo_demanda")
    .reset_index(name="cantidad")
)
series_type_summary["porcentaje"] = (
    series_type_summary["cantidad"] / len(series_2017) * 100
)
series_type_summary

In [ ]:
fig = px.scatter(
    series_2017[series_2017["adi"].notna() & series_2017["cv2"].notna()],
    x="adi",
    y="cv2",
    color="tipo_demanda",
    hover_data=["store_nbr", "family", "pct_ceros", "ventas_totales"],
    log_x=True,
    log_y=True,
    title="Clasificación de demanda por ADI y CV² — 2017",
)
fig.add_vline(x=ADI_THRESHOLD, line_dash="dot")
fig.add_hline(y=CV2_THRESHOLD, line_dash="dot")
fig.update_layout(
    xaxis_title="ADI — intervalo promedio entre demandas",
    yaxis_title="CV² — variabilidad de demanda positiva",
)
fig.show()

In [ ]:
series_rank = (
    series_2017
    .sort_values("ventas_totales", ascending=False)
    .reset_index(drop=True)
)
series_rank["ranking"] = series_rank.index + 1
total_sales_2017 = series_rank["ventas_totales"].sum()
series_rank["participacion_acumulada"] = (
    series_rank["ventas_totales"].cumsum() / total_sales_2017 * 100
)

top_200_share = series_rank.head(200)["ventas_totales"].sum() / total_sales_2017 * 100
print(f"Participación de las 200 series principales: {top_200_share:.2f}%")

In [ ]:
fig = px.line(
    series_rank,
    x="ranking",
    y="participacion_acumulada",
    title="Concentración acumulada de ventas por serie modelable",
)
fig.add_vline(x=200, line_dash="dot")
fig.add_hline(y=top_200_share, line_dash="dot")
fig.update_layout(
    xaxis_title="Ranking de series",
    yaxis_title="Participación acumulada (%)",
)
fig.show()

In [ ]:
# Perfiles estructurales de tiendas sobre el mismo período 2017.
store_sales_2017 = (
    sales_2017
    .groupby("store_nbr", as_index=False)
    .agg(ventas_totales=("sales", "sum"))
    .merge(stores, on="store_nbr", how="left")
)

cluster_profile = (
    store_sales_2017
    .groupby("cluster", as_index=False)
    .agg(
        tiendas=("store_nbr", "nunique"),
        ventas_totales=("ventas_totales", "sum"),
        ventas_promedio_tienda=("ventas_totales", "mean"),
    )
)

city_profile = (
    store_sales_2017
    .groupby("city", as_index=False)
    .agg(
        tiendas=("store_nbr", "nunique"),
        ventas_totales=("ventas_totales", "sum"),
    )
    .sort_values("ventas_totales", ascending=False)
)

display(cluster_profile.sort_values("ventas_promedio_tienda", ascending=False).head(10))
display(city_profile.head(10))

### Hallazgo — heterogeneidad de la demanda

La clasificación ADI + CV² reemplaza la regla arbitraria basada únicamente en porcentaje de ceros. **ADI** mide cuánto tiempo transcurre, en promedio, entre ocurrencias de demanda positiva, mientras que **CV²** mide cuán variable es la magnitud cuando sí existe demanda.

Esto permite distinguir técnicamente series regulares, erráticas, intermitentes y lumpy y será útil en la etapa posterior para seleccionar estrategias de pronóstico adecuadas. Las combinaciones sin demanda positiva durante 2017 se reportan separadamente como `Sin demanda positiva`, sin confundirlas con familias estructuralmente no surtidas.

## 6.12 Próxima etapa: estrategia de validación temporal

El EDA finaliza antes de entrenar o evaluar modelos. En la próxima etapa se utilizará un **corte temporal**, no una partición aleatoria: los últimos **15 días** se reservarán como ventana de validación y todo entrenamiento utilizará exclusivamente información anterior.

El baseline estacional y la métrica RMSLE se trasladan al módulo de modelado (`src/models/baseline.py`) y **no se ejecutan en esta entrega**. De esta manera se mantiene una separación clara entre análisis exploratorio y modelado.

# Síntesis del EDA final

El análisis exploratorio muestra que la demanda de Corporación Favorita presenta una estructura fuertemente temporal, heterogénea y concentrada entre tiendas y familias de productos.

Los ceros estructurales por preapertura y familias no comercializadas se mantienen separados de la demanda real, y los análisis de comportamiento se realizan de manera consistente sobre `sales_modelable`. La tendencia agregada se descompone para distinguir crecimiento de las tiendas comparables, expansión del número de establecimientos y desempeño promedio por tienda activa.

La estacionalidad semanal y mensual, las promociones, el calendario comercial, las transacciones, el petróleo y los eventos extraordinarios fueron evaluados respetando el orden temporal. Promociones, feriados y terremoto incorporan tamaños muestrales e intervalos bootstrap para contextualizar la incertidumbre de los efectos observados.

La estructura tienda–familia se caracteriza mediante **ADI + CV²**, reemplazando umbrales arbitrarios de ceros por una taxonomía técnica de demanda regular, errática, intermitente y lumpy.

Con esta etapa queda preparado un panel temporal coherente para la futura ingeniería de variables y modelado. La validación temporal, el baseline y las métricas de error se abordarán recién en la siguiente etapa del proyecto.